# V2 · Giảm số cột khi tìm đường nối panorama
**Vũ Trần Phúc · 23120333 · Nhóm 22**

Chọn runtime GPU trong Colab. Ở ô cấu hình, đặt `SOURCE_PROJECT` trỏ tới bản nhánh V2 trên Drive chứa `src`, `tools` và dữ liệu nhóm (8 ảnh raw hoặc 24 PNG prepared). Notebook sao chép sang thư mục tạm mới rồi build một binary cho V0, V1 và V2. Kết quả được đóng gói ở cuối.

### Phương pháp và frontend
Hai ảnh RGB được làm xám và lọc Gaussian tách ngang/dọc. FAST-9 phát hiện góc, NMS 3×3 và Top-K giữ tối đa 1.500 điểm theo response giảm rồi tọa độ tăng. BRIEF ghi 256 phép so sánh cường độ. V1/V2 dùng một thread mỗi pixel cho khâu xử lý ảnh, CUB radix sort cho Top-K và một thread mỗi descriptor. Matcher dùng Hamming, ratio 0,75 và kiểm tra hai chiều. RANSAC ước lượng H từ phải sang trái. Inverse warp, bù sáng và seam/feather tạo panorama.

### Lý do chọn stage 6
Notebook V1 đã lưu trên Tesla T4 cho thấy matching trung bình 0,786/1,084 ms ở medium/large, trong khi warp và blend cần 10,362/32,760 ms. Đây là timing toàn stage. V2 khảo sát riêng phần seam. Chi phí seam chỉ xác định tại pixel mà cả A và B cùng phủ. V1 cấp phát và duyệt toàn chiều rộng canvas, kể cả các cột chắc chắn ở ngoài A.

### Thay đổi
V2 dùng miền cột từ floor(offset_x) đến min(canvas_w, ceil(offset_x) + width_A), loại trừ đầu phải. Kernel cost dùng block 16×16. Quy hoạch động dùng một block 1.024 thread xử lý lần lượt từng hàng. Mỗi cột lưu chi phí float và hướng truy vết int8; hai hàng tổng tích lũy dùng double. Với H hàng và W cột thay vì C cột canvas, phần bộ nhớ yêu cầu cho ba buffer này giảm (C−W)×(5H+16) byte. Cấp phát được tái sử dụng giữa các lượt. Khi truy vết, cộng lại cột đầu để lấy tọa độ canvas. Phần feather đi qua cùng luồng tính như V1.

### Giả thuyết
Canvas rộng và bật seam có thể hưởng lợi. Khi A chiếm gần hết chiều rộng hoặc dùng feather, ít cơ hội giảm thời gian. Chi phí truyền dữ liệu và các stage khác giới hạn mức cải thiện end-to-end. Các ô dưới đo lại trong cùng phiên, xuất cả trường hợp chậm hơn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import zipfile

SOURCE_PROJECT = Path("/content/drive/Othercomputers/My Laptop/image-stitching")
RUNS = 5
WARMUP = 1

CUDA_ARCH = None  # Tự nhận từ GPU; dùng "sm_75" để chỉ định T4.

if not SOURCE_PROJECT.is_dir():
    raise FileNotFoundError(f"Không thấy {SOURCE_PROJECT}. Kiểm tra ô mount và đường dẫn Drive.")
required = ("src/gpu_pipeline.cu", "tools/verify_v1.py", "tools/benchmark.py", "requirements.txt", "dataset")
missing = [name for name in required if not (SOURCE_PROJECT / name).exists()]
if missing:
    raise FileNotFoundError(f"Drive chưa có các mục: {', '.join(missing)}. Đợi đồng bộ mã và dữ liệu.")
if RUNS < 5 or WARMUP < 1:
    raise ValueError("Thực nghiệm cần ít nhất 5 lần đo và 1 warmup.")

def capture(*cmd):
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT).strip()

if not shutil.which("nvidia-smi") or not shutil.which("nvcc"):
    raise RuntimeError("Chọn GPU runtime có NVIDIA CUDA Toolkit (nvidia-smi và nvcc).")
try:
    gpu_info = capture("nvidia-smi", "-L")
    capability = capture("nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader").splitlines()[0]
except (subprocess.CalledProcessError, IndexError) as error:
    raise RuntimeError("Colab chưa nhận được GPU. Kiểm tra lại loại runtime.") from error

if CUDA_ARCH is None:
    CUDA_ARCH = "sm_" + capability.strip().replace(".", "")
nvcc_info = capture("nvcc", "--version")
print(gpu_info, "\n", nvcc_info, "\nCUDA_ARCH:", CUDA_ARCH, sep="")

ROOT = Path(tempfile.mkdtemp(prefix="group22-v2-", dir="/content"))
ignore = shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints", "*.exe", "*.obj", "*.o")
print("Đang sao chép dự án từ:", SOURCE_PROJECT, flush=True)
for name in ("src", "tools"):
    shutil.copytree(SOURCE_PROJECT / name, ROOT / name, ignore=ignore)
shutil.copy2(SOURCE_PROJECT / "requirements.txt", ROOT / "requirements.txt")
(ROOT / "dataset").mkdir()
for name in ("raw", "prepared", "synthetic"):
    source_folder = SOURCE_PROJECT / "dataset" / name
    if source_folder.is_dir():
        shutil.copytree(source_folder, ROOT / "dataset" / name, ignore=ignore)
    else:
        (ROOT / "dataset" / name).mkdir()

digest = hashlib.sha256()
for item in sorted(ROOT.rglob("*")):
    if item.is_file():
        digest.update(item.relative_to(ROOT).as_posix().encode("utf-8") + b"\0")
        file_digest = hashlib.sha256()
        with item.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                file_digest.update(block)
        digest.update(file_digest.digest())

EVIDENCE = ROOT / "experiment"
EVIDENCE.mkdir()
ENV = {**os.environ, "NO_COLOR": "1"}
metadata = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "source_project": str(SOURCE_PROJECT),
    "snapshot_sha256": digest.hexdigest(),
    "gpu": gpu_info,
    "cuda_arch": CUDA_ARCH,
    "nvcc": nvcc_info,
    "python": sys.version,
    "platform": platform.platform(),
    "runs": RUNS,
    "warmup": WARMUP,
}
(EVIDENCE / "session.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Thư mục chạy:", ROOT)

def run(*args, check=True, cwd=ROOT):
    return subprocess.run([str(arg) for arg in args], cwd=cwd, env=ENV, check=check).returncode

In [ ]:
from IPython.display import display

run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas")
from PIL import Image
sys.path.insert(0, str(ROOT / "tools"))
from benchmark import SITES

RESOLUTIONS = ("small", "medium", "large")
prepared = [ROOT / "dataset/prepared" / size / f"{site}_{side}.png"
            for size in RESOLUTIONS for site in SITES for side in ("left", "right")]
synthetic_root = ROOT / "dataset/synthetic"
synthetic_files = [synthetic_root / name for name in
                   ("synthetic_left.png", "synthetic_right.png", "H_ground_truth.txt")]
if any(not path.is_file() for path in prepared):
    run(sys.executable, "tools/prepare_dataset.py")
elif any(not path.is_file() for path in synthetic_files):
    run(sys.executable, "tools/prepare_dataset.py", "--synthetic-only")
missing = [str(path.relative_to(ROOT)) for path in prepared + synthetic_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Thiếu ảnh thực nghiệm: {', '.join(missing)}")

run(sys.executable, "tools/build.py", "--cuda", "--arch", CUDA_ARCH)
BINARY = ROOT / "stitch"

## Đối chiếu và đo thời gian
Mỗi cấu hình có một warmup và năm lượt đo. Kiểm tra V2/V1 yêu cầu match, inlier, H, seam và pixel trùng nhau. So với V0 dùng ngưỡng của V1. Ca nhân tạo kiểm tra hai chiều dịch, offset lẻ và tie-break trên vùng chi phí đồng đều.

In [ ]:
checks = {}
def record(name, *args, cwd=ROOT):
    checks[name] = run(*args, cwd=cwd, check=False)

record("build_backend_test", "make", "test-backend", f"CUDA_ARCH={CUDA_ARCH}", cwd=ROOT/"src")
if checks["build_backend_test"] == 0:
    record("compact_seam", ROOT/"src/test_backend", "--check-compact-seam")
for reference in (0, 1):
    name = f"verify_{reference}_2"
    record(name, sys.executable, "tools/verify_v1.py", "--reference", reference, "--candidate", 2,
           "--report", EVIDENCE/f"{name}.json")
for version in (0, 1, 2):
    record(f"benchmark_{version}", sys.executable, "tools/benchmark.py", "--version", version,
           "--runs", RUNS, "--warmup", WARMUP)
(EVIDENCE/"exit_codes.json").write_text(json.dumps(checks, indent=2))
# Collect CUDA kernel traces separately from the benchmark samples.
profiler = shutil.which("nsys")
profile_status = {}
for version in (1, 2):
    name = f"large_challenging_v{version}"
    if not profiler:
        profile_status[name] = "nsys unavailable"
        continue
    folder = ROOT / "dataset/prepared/large"
    cmd = [profiler, "profile", "--trace=cuda", "--sample=none", "--cpuctxsw=none",
           "-o", str(EVIDENCE/name), str(BINARY), str(folder/"site4_challenging_left.png"),
           str(folder/"site4_challenging_right.png"), str(ROOT/f"{name}.png"),
           "--version", str(version), "--warmup", "1", "--runs", "1"]
    profile = subprocess.run(cmd, cwd=ROOT, env=ENV, capture_output=True, text=True)
    (EVIDENCE/f"{name}.log").write_text(profile.stdout + profile.stderr)
    profile_status[name] = profile.returncode
    if profile.returncode == 0:
        stats = subprocess.run([profiler, "stats", "--report", "cuda_gpu_kern_sum",
                                str(EVIDENCE/f"{name}.nsys-rep")], capture_output=True, text=True)
        (EVIDENCE/f"{name}_kernels.txt").write_text(stats.stdout + stats.stderr)
        profile_status[name + "_stats"] = stats.returncode
(EVIDENCE/"profiler_status.json").write_text(json.dumps(profile_status, indent=2))
print("Profiler:", profile_status)
print(checks)

In [ ]:
import pandas as pd

def load_report(path):
    return json.loads(path.read_text()) if path.is_file() else {}

def cases(version):
    report = load_report(ROOT/f"outputs_v{version}"/"summary.json")
    real = {f"{r['resolution']}/{r['site']}": r for r in report.get("results", [])}
    return {**real, "synthetic": report.get("correctness", {})}

v0, v1, v2 = map(cases, (0, 1, 2))
cpu_check, gpu_check = [load_report(EVIDENCE/f"verify_{v}_2.json") for v in (0, 1)]
rows = []
for case in [f"{size}/{site}" for size in RESOLUTIONS for site in SITES] + ["synthetic"]:
    runs = [v.get(case, {}) for v in (v0, v1, v2)]
    verified = all(check.get(case, {}).get("passed") for check in (cpu_check, gpu_check))
    row = {"case": case, "verified": verified, "seam_used": gpu_check.get(case, {}).get("seam_used"),
           "descriptors_A_B": runs[1].get("descriptor_counts")}
    for version, result in enumerate(runs):
        for metric in ("matching_ms", "warp_blend_ms", "total_pipeline_ms"):
            row[f"v{version}_{metric}"] = result.get("mean", {}).get(metric)
    if verified and all(r.get("success") for r in runs):
        row["stage6_speedup_v1_v2"] = row["v1_warp_blend_ms"] / row["v2_warp_blend_ms"]
        for version in (0, 1):
            row[f"pipeline_speedup_v{version}_v2"] = row[f"v{version}_total_pipeline_ms"] / row["v2_total_pipeline_ms"]
    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(EVIDENCE/"v2_comparison.csv", index=False)
display(table.round(3))
valid = table[table.verified].dropna(subset=["v2_total_pipeline_ms"])
if "pipeline_speedup_v1_v2" in valid:
    display(valid.groupby("seam_used")[["stage6_speedup_v1_v2", "pipeline_speedup_v1_v2"]].agg(["count", "median"]))
    for _, r in valid.dropna(subset=["pipeline_speedup_v1_v2"]).iterrows():
        change = r.pipeline_speedup_v1_v2
        print(f"{r['case']}: V2 {'nhanh hơn' if change > 1 else 'chậm hơn' if change < 1 else 'bằng V1'} (V1/V2 = {change:.3f}).")

## Lưu bằng chứng
ZIP gồm các bảng, kết quả đối chiếu, CSV từng lượt, ảnh và log. Nếu một kiểm tra lỗi, notebook vẫn tạo ZIP trước khi báo lỗi.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archive = Path(f"/content/Group22_V2_{stamp}.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for folder in (EVIDENCE, ROOT / "outputs_v0", ROOT / "outputs_v1", ROOT / "outputs_v2"):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=path.relative_to(ROOT))
print(f"Tải tệp trong tab Files: {archive}")
# from google.colab import files
# files.download(str(archive))
if any(checks.values()) or not all(r["verified"] for r in rows):
    raise RuntimeError(f"Thực nghiệm còn lỗi. Xem {archive} và exit_codes.json.")

### Nguồn và khai báo AI
V2 kế thừa pipeline, hàm chi phí đường nối và tie-break từ mã V1 của nhóm tại commit `5a320ee`. Số liệu định hướng lấy từ output đã lưu trong `Group22_V0_V1_Colab.ipynb`. Tài liệu thuật toán nền cho FAST và BRIEF được trích ở đầu `src/v0_frontend.cpp`.
Codex hỗ trợ đọc phân công, đề xuất miền cột, viết thay đổi CUDA, ca kiểm tra và notebook. Các bảng thực nghiệm V2 lấy từ output của phiên Colab chạy notebook.